# Intégration de logos pour la détection des bacs de tri

Le but est d'avoir trois logos (un pour chaque catégorie de déchets) 

## 1. Import

In [1]:
import cv2
import matplotlib.pyplot as plt
import os
import numpy as np
import albumentations as A
import time
from tqdm import tqdm
import random
from pathlib import Path
import glob

## 2. PATH

In [11]:
LOGO_DIR = Path("../data/logos")
BG_DIR = Path("../dataset/TACO-master/data/images")
OUTPUT_DIR = Path("../dataset/synthetic_logos_yolo")

# Création des dossiers de sortie pour YOLO
(OUTPUT_DIR / "train/images").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "train/labels").mkdir(parents=True, exist_ok=True)

# On crée la liste des fichiers d'arrière-plan (obligatoire pour le random.choice)
all_backgrounds = list(BG_DIR.glob("*.jpg"))

print(f"Nombre de fonds disponibles : {len(all_backgrounds)}")

Nombre de fonds disponibles : 1500


## 3. Changer la couleur des logos si nécessaire

Ce code va écraser/remplacer les anciens logos par les nouveaux

In [19]:
import os
from PIL import Image

# 1. Chemins des dossiers (adapte les noms si nécessaire)
folder = "../data/logos"  # Dossier où se trouvent tes images téléchargées


# 2. Liste des formats d'images acceptés
valid_extensions = (".png", ".jpg", ".jpeg", ".webp")
    
# Récupération de tous les fichiers du dossier
files = [f for f in os.listdir(folder) if f.lower().endswith(valid_extensions)]
    
print(f"Trouvé : {len(files)} image(s) à traiter.\n")

for filename in files:
    try:
        # Chemin complet du fichier source
        input_path = os.path.join(folder, filename)
            
        # Ouverture de l'image et conversion en RGBA
        img = Image.open(input_path).convert("RGBA")
            
        # Extraction du canal alpha (la forme/silhouette du logo)
        alpha = img.split()[-1]
            
        # Création d'une image noire de la même taille
        black_img = Image.new("RGBA", img.size, (0, 0, 0, 255))
            
        # Application du masque alpha sur le fond noir
        final_img = Image.new("RGBA", img.size, (0, 0, 0, 0))
        final_img.paste(black_img, (0, 0), mask=alpha)
            
        # Sauvegarde dans le dossier de sortie (on force le format PNG pour garder la transparence)
        name_without_ext = os.path.splitext(filename)[0]
        output_filename = f"{name_without_ext}.png"
        output_path = os.path.join(folder, output_filename)

        # Fermer l'image d'origine pour libérer le fichier avant d'écraser/supprimer
        img.close()
        
        final_img.save(output_path, "PNG")
            
        # Si l'ancien fichier n'était pas un .png (ex: .jpg), on le supprime pour éviter les doublons
        if filename.lower() != output_filename.lower():
            os.remove(input_path)
            print(f"✓ {filename} -> Remplacé par {output_filename} (noir).")
        else:
            print(f"✓ {filename} -> Écrasé en noir avec succès.")
            
    except Exception as e:
        print(f"✗ Erreur lors du traitement de {filename} : {e}")

print(f"\nTraitement terminé ! Les images d'origine dans '{folder}' ont été remplacées par les versions noires.")

Trouvé : 3 image(s) à traiter.

✓ logo_jaune.png -> Écrasé en noir avec succès.
✓ logo_noir.png -> Écrasé en noir avec succès.
✓ logo_verre.png -> Écrasé en noir avec succès.

Traitement terminé ! Les images d'origine dans '../data/logos' ont été remplacées par les versions noires.


## 4. Charger les logos de tri

In [12]:
# Dictionnaire pour tes 3 catégories
logos_pilotes = {
    "verre": cv2.imread(str(LOGO_DIR / "logo_verre.png"), cv2.IMREAD_UNCHANGED),
    "jaune": cv2.imread(str(LOGO_DIR / "logo_jaune.png"), cv2.IMREAD_UNCHANGED),
    "noir":  cv2.imread(str(LOGO_DIR / "logo_noir.png"), cv2.IMREAD_UNCHANGED)
}

Vérification, si canaux=4 alors on a bien un fond transparent

In [13]:
for nom, img in logos_pilotes.items():
    if img is None:
        print(f"⚠️ Erreur : Le logo '{nom}' n'a pas été trouvé. Vérifie le chemin !")
    else:
        print(f"✅ Logo '{nom}' chargé : {img.shape[1]}x{img.shape[0]} pixels (Canaux: {img.shape[2]})")

✅ Logo 'verre' chargé : 211x200 pixels (Canaux: 4)
✅ Logo 'jaune' chargé : 302x312 pixels (Canaux: 4)
✅ Logo 'noir' chargé : 330x317 pixels (Canaux: 4)


## 5. Modification sur les logos

In [21]:
def apply_scl_transform(logo, target_width):
    # On redimensionne le logo pour qu'il fasse, par exemple, 
    # entre 10% et 30% de la largeur du fond (target_width)
    scale = random.uniform(0.1, 0.3)
    new_w = int(target_width * scale)
    
    h, w = logo.shape[:2]
    new_h = int(h * (new_w / w))
    
    logo = cv2.resize(logo, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
    # On garde le reste des transformations (rotation, couleur)
    rows, cols = logo.shape[:2]
    M_rot = cv2.getRotationMatrix2D((cols/2, rows/2), random.randint(-15, 15), 1)
    logo = cv2.warpAffine(logo, M_rot, (cols, rows), borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0,0))

    # On crée une couleur BGR aléatoire 
    random_color = [random.randint(0, 255) for _ in range(3)]

    # On détecte les pixels qui ne sont pas transparents (canal Alpha > 0)
    alpha_channel = logo[:, :, 3]
    mask = alpha_channel > 0
    
    # On applique la couleur aléatoire sur ces pixels
    for c in range(3): # Canaux 0=Bleu, 1=Vert, 2=Rouge
        logo[mask, c] = random_color[c]
    
    r = random.uniform(0.7, 1.3)
    logo[:, :, :3] = np.clip(logo[:, :, :3].astype(np.float32) * r, 0, 255).astype(np.uint8)
    
    return logo

## Transformation SCL (Synthetic Context Logo)

In [22]:
def create_synthetic_image(background_path, logo_img):
    bg = cv2.imread(str(background_path))
    if bg is None: return None, None
    bg_h, bg_w = bg.shape[:2]

    # Appliquer les transformations SCL au logo (Géométrie + Couleur), bg_w est utilisé pour redimensionner le logo à une taille réaliste par rapport au fond
    logo = apply_scl_transform(logo_img, bg_w) 
    l_h, l_w = logo.shape[:2]

    # Sécurité : si après transformation le logo est toujours trop grand (rare)
    if l_w >= bg_w or l_h >= bg_h:
        return None, None

    x = random.randint(0, bg_w - l_w)
    y = random.randint(0, bg_h - l_h)

    # 3. Fusionner le logo avec le fond en utilisant le canal Alpha
    alpha_mask = logo[:, :, 3] / 255.0
    for c in range(3):
        bg[y:y+l_h, x:x+l_w, c] = (1.0 - alpha_mask) * bg[y:y+l_h, x:x+l_w, c] + alpha_mask * logo[:, :, c]

    # 4. Coordonnées YOLO
    x_center = (x + l_w / 2) / bg_w
    y_center = (y + l_h / 2) / bg_h
    w_norm = l_w / bg_w
    h_norm = l_h / bg_h
    
    return bg, [x_center, y_center, w_norm, h_norm]

## Génération des images

In [ ]:
# Paramètres de ton projet
N_IMAGES_PER_CLASS = 500
# Associe chaque nom à un index (0, 1, 2) pour YOLO
class_map = {"verre": 0, "jaune": 1, "noir": 2}

for label_name, logo_orig in logos_pilotes.items():
    print(f"Génération de la classe : {label_name}...")
    class_id = class_map[label_name]
    
    for i in range(N_IMAGES_PER_CLASS):
        # 1. Choisir un fond TACO au hasard 
        bg_path = random.choice(all_backgrounds)
        img_synth, bbox = create_synthetic_image(bg_path, logo_orig)
        
        if img_synth is None: continue
        
        # Nom de fichier unique
        file_name = f"{label_name}_synth_{i}"
        
        # 2. Sauvegarder l'image
        cv2.imwrite(str(OUTPUT_DIR / "train/images" / f"{file_name}.jpg"), img_synth)
        
        # 3. Sauvegarder le label YOLO (.txt)
        # Format : <class_id> <x_center> <y_center> <width> <height>
        with open(OUTPUT_DIR / "train/labels" / f"{file_name}.txt", "w") as f:
            f.write(f"{class_id} {bbox[0]} {bbox[1]} {bbox[2]} {bbox[3]}")

print("✅ Dataset synthétique généré avec succès !")

Génération de la classe : verre...
Génération de la classe : jaune...
Génération de la classe : noir...
✅ Dataset synthétique généré avec succès !


## Création du .yaml

In [17]:
import yaml

base_path = '../dataset/synthetic_logos_yolo'

data_yaml = {
        'path': base_path,
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'nc': len(class_map),
        'names': class_map,
    }

yaml_file_path = f'{base_path}/data_logos.yaml'
with open(yaml_file_path, 'w') as outfile:
    yaml.dump(data_yaml, outfile, default_flow_style=False)

print(f"Fichier YAML créé ici : {yaml_file_path}")

Fichier YAML créé ici : ../dataset/synthetic_logos_yolo/data_logos.yaml
